In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error
import shutil
import os
from google.colab import files
import warnings

warnings.filterwarnings('ignore')
plt.rcParams["axes.axisbelow"] = True
RANDOM_STATE = 123

def save_local(data_or_fig, filename, is_plot=False):
    path = Path("output_robustness") / filename
    path.parent.mkdir(parents=True, exist_ok=True)
    if is_plot:
        plt.tight_layout()
        data_or_fig.savefig(path, dpi=400)
        pdf_path = path.with_suffix(".pdf")
        data_or_fig.savefig(pdf_path, dpi=250)
        plt.close(data_or_fig)
    else:
        data_or_fig.to_csv(path, index=False)

# =====================================================================
# FINAL MODEL PREPROCESSING (Directly from model.ipynb)
# =====================================================================
def get_class_preprocessor(num_cols, cat_cols):
    """Classification uses the standard ColumnTransformer from your final model."""
    return ColumnTransformer(transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
        ('cat', Pipeline(steps=[('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))]), cat_cols)
    ])

def preprocess_prediction_features(X_train, X_test):
    """Prediction uses the exact manual processing function from your final model."""
    num_cols = X_train.select_dtypes(include=['int64', 'float64', 'Int64']).columns.tolist()
    cat_cols = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

    num_imputer = SimpleImputer(strategy="median")
    cat_imputer = SimpleImputer(strategy="most_frequent")

    X_train_num = pd.DataFrame(num_imputer.fit_transform(X_train[num_cols]), columns=num_cols, index=X_train.index)
    X_test_num = pd.DataFrame(num_imputer.transform(X_test[num_cols]), columns=num_cols, index=X_test.index)

    if cat_cols:
        X_train_cat = pd.DataFrame(cat_imputer.fit_transform(X_train[cat_cols]), columns=cat_cols, index=X_train.index)
        X_test_cat = pd.DataFrame(cat_imputer.transform(X_test[cat_cols]), columns=cat_cols, index=X_test.index)
        encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
        X_train_encoded = pd.DataFrame(encoder.fit_transform(X_train_cat), columns=encoder.get_feature_names_out(cat_cols), index=X_train.index)
        X_test_encoded = pd.DataFrame(encoder.transform(X_test_cat), columns=encoder.get_feature_names_out(cat_cols), index=X_test.index)
    else:
        X_train_encoded = pd.DataFrame(index=X_train.index)
        X_test_encoded = pd.DataFrame(index=X_test.index)

    scaler = StandardScaler()
    X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_num), columns=num_cols, index=X_train.index)
    X_test_scaled = pd.DataFrame(scaler.transform(X_test_num), columns=num_cols, index=X_test.index)

    X_train_processed = pd.concat([X_train_scaled, X_train_encoded], axis=1)
    X_test_processed = pd.concat([X_test_scaled, X_test_encoded], axis=1)

    return X_train_processed, X_test_processed

# =====================================================================
# TUNING & TRAINING
# =====================================================================
def tune_classification_once(df, features, target):
    """Finds best hyperparameters for Logistic Regression before robustness checks."""
    X, y = df[features], df[target]
    num_cols = X.select_dtypes(include=['number']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

    pipe = Pipeline(steps=[
        ('preprocessor', get_class_preprocessor(num_cols, cat_cols)),
        ('classifier', LogisticRegression(random_state=RANDOM_STATE))
    ])

    search = GridSearchCV(pipe, param_grid={'classifier__C': [0.1, 1.0, 10.0]},
                          scoring='accuracy', cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE))
    search.fit(X, y)
    return search.best_params_['classifier__C']

def tune_prediction_once(df, features, target):
    """Finds best hyperparameters for Random Forest before robustness checks."""
    X, y = df[features], df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
    X_train_processed, _ = preprocess_prediction_features(X_train, X_test)

    search = GridSearchCV(RandomForestRegressor(random_state=RANDOM_STATE),
                          param_grid={"n_estimators": [50, 100], "max_depth": [3, 5]},
                          scoring='neg_root_mean_squared_error', cv=KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE))
    search.fit(X_train_processed, y_train)
    return search.best_params_

def train_and_evaluate(df, features, target, best_params, random_state, is_class):
    """Trains and evaluates the model using the exact final pipeline architecture."""
    X, y = df[features], df[target]

    if is_class:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=random_state)
        num_cols = X.select_dtypes(include=['number']).columns.tolist()
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

        model = LogisticRegression(C=best_params, random_state=RANDOM_STATE)
        pipe = Pipeline(steps=[('preprocessor', get_class_preprocessor(num_cols, cat_cols)), ('classifier', model)])
        pipe.fit(X_train, y_train)
        return float(accuracy_score(y_test, pipe.predict(X_test)))
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
        X_train_proc, X_test_proc = preprocess_prediction_features(X_train, X_test)

        model = RandomForestRegressor(n_estimators=best_params['n_estimators'], max_depth=best_params['max_depth'], random_state=RANDOM_STATE)
        model.fit(X_train_proc, y_train)
        return float(np.sqrt(mean_squared_error(y_test, model.predict(X_test_proc))))

# =====================================================================
# ROBUSTNESS CHECKS
# =====================================================================
def run_all_checks(df, features, target, best_params, is_class):
    rows = []

    # 1. Data Size Robustness
    for frac in [0.4, 0.6, 0.8, 1.0]:
        sample_n = max(20, int(round(frac * len(df))))
        for rep in range(3):
            seed = RANDOM_STATE + rep
            sampled = df.sample(n=sample_n, replace=False, random_state=seed)
            metric = train_and_evaluate(sampled, features, target, best_params, seed, is_class)
            rows.append({"check": "data_size", "train_fraction": frac, "metric": metric})

    # 2. Missing Data Robustness
    for rate in [0.10, 0.20, 0.30]:
        for rep in range(3):
            seed = RANDOM_STATE + rep
            rng = np.random.default_rng(seed)
            df_miss = df.copy()
            for col in features:
                mask = rng.random(len(df_miss)) < rate
                df_miss.loc[mask, col] = np.nan
            metric = train_and_evaluate(df_miss, features, target, best_params, seed, is_class)
            rows.append({"check": "missing_data", "missing_rate": rate, "metric": metric})

    # 3. Perturbation Robustness
    num_cols = df[features].select_dtypes(include=['number']).columns.tolist()
    for scale in [0.025, 0.05, 0.10]:
        for rep in range(3):
            seed = RANDOM_STATE + rep
            rng = np.random.default_rng(seed)
            df_pert = df.copy()
            for col in num_cols:
                std = df_pert[col].std()
                if pd.notna(std) and std > 0:
                    df_pert[col] += rng.normal(0.0, scale * std, len(df_pert))
            metric = train_and_evaluate(df_pert, features, target, best_params, seed, is_class)
            rows.append({"check": "perturbation", "noise_scale_sd": scale, "metric": metric})

    # 4. Bootstrap Robustness
    for b in range(20):
        seed = RANDOM_STATE + b
        boot = df.sample(frac=1.0, replace=True, random_state=seed)
        metric = train_and_evaluate(boot, features, target, best_params, seed, is_class)
        rows.append({"check": "bootstrap", "bootstrap_id": b, "metric": metric})

    return pd.DataFrame(rows)

# =====================================================================
# PLOTTING
# =====================================================================
def plot_robustness(combined_df, prefix, metric_name, model_label):
    # Data Size
    data_size = combined_df[combined_df['check'] == 'data_size'].groupby('train_fraction')['metric'].agg(['mean', 'std']).reset_index()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.errorbar(data_size['train_fraction'], data_size['mean'], yerr=data_size['std'], marker="o", capsize=4, color="#4c72b0")
    ax.set_xlabel("Training Data Fraction")
    ax.set_ylabel(metric_name.upper())
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    save_local(fig, f"{prefix}_robustness_data_size.png", is_plot=True)

    # Missing Data
    missing = combined_df[combined_df['check'] == 'missing_data'].groupby('missing_rate')['metric'].agg(['mean', 'std']).reset_index()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.errorbar(missing['missing_rate'], missing['mean'], yerr=missing['std'], marker="o", capsize=4, color="#4c72b0")
    ax.set_xlabel("Missing Data Fraction")
    ax.set_ylabel(metric_name.upper())
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    save_local(fig, f"{prefix}_robustness_missing_data.png", is_plot=True)

    # Perturbation
    perturb = combined_df[combined_df['check'] == 'perturbation'].groupby('noise_scale_sd')['metric'].agg(['mean', 'std']).reset_index()
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.errorbar(perturb['noise_scale_sd'], perturb['mean'], yerr=perturb['std'], marker="o", capsize=4, color="#4c72b0")
    ax.set_xlabel("Noise Scale (SD units)")
    ax.set_ylabel(metric_name.upper())
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    save_local(fig, f"{prefix}_robustness_perturbation.png", is_plot=True)

    # Bootstrap Boxplot
    boot = combined_df[combined_df['check'] == 'bootstrap']
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.boxplot(boot['metric'].dropna().values, tick_labels=[model_label])
    ax.set_ylabel(metric_name.upper())
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    save_local(fig, f"{prefix}_robustness_bootstrap_boxplot.png", is_plot=True)

    # Bootstrap Histogram
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.hist(boot['metric'].dropna(), bins=10, alpha=0.7, color="#4c72b0", edgecolor='black')
    ax.set_xlabel(metric_name.upper())
    ax.set_ylabel("Frequency")
    ax.grid(axis="y", linestyle="--", alpha=0.7)
    save_local(fig, f"{prefix}_robustness_bootstrap_histogram.png", is_plot=True)


# =====================================================================
# MAIN EXECUTION
# =====================================================================
def main():
    os.makedirs("output_robustness", exist_ok=True)

    print("Fetching Processed Data...")
    df_class = pd.read_csv("https://drive.google.com/uc?export=download&id=1tW6Si8XiQJclYrP_1be6Td7tU7DbLyGG")
    df_pred = pd.read_csv("https://drive.google.com/uc?export=download&id=13vSsB9ys6R4eumq6Z51Ha7MpEDHt7M6u")

    # Format Target
    if df_class['Foundation Type'].dtype == 'object':
        df_class['Foundation Type'] = df_class['Foundation Type'].map({'Deep': 1, 'Shallow': 0})
    df_class.dropna(subset=['Foundation Type'], inplace=True)
    df_pred.dropna(subset=['Pier total Depth (ft)'], inplace=True)

    # --- CLASSIFICATION ---
    print(f"\n{'='*60}\n--- Running Robustness: CLASSIFICATION TASK ---\n{'='*60}")
    feat_c = df_class.drop(columns=['Foundation Type']).columns.tolist()

    print("  -> Tuning Logistic Regression via GridSearchCV...")
    best_c = tune_classification_once(df_class, feat_c, 'Foundation Type')
    print(f"  -> Best Parameter Found: C={best_c}")

    print("  -> Running Stress Tests...")
    combined_c = run_all_checks(df_class, feat_c, 'Foundation Type', best_c, is_class=True)
    save_local(combined_c, "class_robustness_all_runs.csv", is_plot=False)
    plot_robustness(combined_c, "class", "accuracy", "Logistic Regression")

    # --- PREDICTION ---
    print(f"\n{'='*60}\n--- Running Robustness: PREDICTION TASK ---\n{'='*60}")
    feat_p = df_pred.drop(columns=['Pier total Depth (ft)']).columns.tolist()

    print("  -> Tuning Random Forest via GridSearchCV...")
    best_p = tune_prediction_once(df_pred, feat_p, 'Pier total Depth (ft)')
    print(f"  -> Best Parameters Found: {best_p}")

    print("  -> Running Stress Tests...")
    combined_p = run_all_checks(df_pred, feat_p, 'Pier total Depth (ft)', best_p, is_class=False)
    save_local(combined_p, "pred_robustness_all_runs.csv", is_plot=False)
    plot_robustness(combined_p, "pred", "rmse", "Random Forest")

    # --- DOWNLOAD ---
    print("\n========================================================")
    print("Zipping all generated robustness files into a single archive...")
    shutil.make_archive("all_robustness_results", 'zip', "output_robustness")
    print("Downloading all_robustness_results.zip...")
    files.download("all_robustness_results.zip")

if __name__ == "__main__":
    main()


Fetching Processed Data...

--- Running Robustness: CLASSIFICATION TASK ---
  -> Tuning Logistic Regression via GridSearchCV...
  -> Best Parameter Found: C=1.0
  -> Running Stress Tests...

--- Running Robustness: PREDICTION TASK ---
  -> Tuning Random Forest via GridSearchCV...
  -> Best Parameters Found: {'max_depth': 5, 'n_estimators': 100}
  -> Running Stress Tests...

Zipping all generated robustness files into a single archive...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>